# DuST-DINOv3 trên Google Colab

Chạy đủ pipeline: **làm sạch dữ liệu UAH → Stage 1 → Stage 2 → Stage 3 → Stage 4**.

- Code đọc trực tiếp từ Google Drive (thư mục `DUST/dust_dinov3`), nên sửa code trên VS Code xong, đợi Drive đồng bộ là chạy lại được.
- Dữ liệu được chép sang ổ local của Colab (`/content`) để đọc nhanh.
- Checkpoint, log, video feature (và tuỳ chọn cả pickle dữ liệu) được lưu vào `dust_dinov3/runs/<RUN_NAME>/` trên Drive, không mất khi runtime bị ngắt.

**Trước khi chạy**: Runtime → Change runtime type → chọn GPU (T4). Đảm bảo Drive đã đồng bộ xong thư mục `dust_dinov3/` (có `model.safetensors`) và đủ 40 video của UAH-DriveSet.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 0. Cấu hình đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

DRIVE = '/content/drive/MyDrive'
# Tìm project: ưu tiên thư mục E:\Chuyên đề được Drive for desktop đồng bộ từ máy tính
# (Colab thấy ở /content/drive/Othercomputers/<tên máy>/...), sau đó mới tới bản trong My Drive.
PROJECT_CANDIDATES = (sorted(glob.glob('/content/drive/Othercomputers/*/DUST')) +
                      sorted(glob.glob('/content/drive/Othercomputers/*/Chuyên đề/DUST')) + [f'{DRIVE}/Chuyên đề/DUST'])
PROJECT_DIR = next((c for c in PROJECT_CANDIDATES if os.path.isdir(f'{c}/dust_dinov3')), PROJECT_CANDIDATES[-1])
print('Các vị trí đã thử:', PROJECT_CANDIDATES)
CODE_DIR = f'{PROJECT_DIR}/dust_dinov3'
# Nếu clone thẳng repo Checking-Driver lên Drive (không dùng bố cục DUST/dust_dinov3), sửa 2 dòng dưới:
# CODE_DIR = f'{DRIVE}/Checking-Driver'
# PROJECT_DIR = f'{DRIVE}'   # thư mục chứa UAH-DRIVESET-v1

# UAH-DriveSet có thể nằm trong DUST/ hoặc cạnh DUST/ trên Drive
UAH_CANDIDATES = [f'{PROJECT_DIR}/UAH-DRIVESET-v1', f'{os.path.dirname(PROJECT_DIR)}/UAH-DRIVESET-v1']

RUN_NAME = 'dinov3_vits16_run1'                  # đổi tên cho mỗi lần thử nghiệm
RUN_DIR = f'{CODE_DIR}/runs/{RUN_NAME}'          # trên Drive: checkpoint, log, feature
DRIVE_DATA_DIR = f'{RUN_DIR}/data'               # bản lưu pickle dữ liệu trên Drive

LOCAL_UAH = '/content/uah'                       # bản chép local của UAH-DriveSet
LOCAL_DATA = '/content/data'                     # pickle dữ liệu dùng để train

# ---- kiểm tra
assert os.path.isdir(CODE_DIR), f'Không thấy {CODE_DIR}. Drive đã đồng bộ thư mục dust_dinov3 chưa?'
assert os.path.isfile(f'{CODE_DIR}/model.safetensors'), 'Thiếu dust_dinov3/model.safetensors trên Drive'
UAH_DIR = next((c for c in UAH_CANDIDATES if os.path.isdir(c)), None)
assert UAH_DIR, f'Không thấy UAH-DRIVESET-v1 ở {UAH_CANDIDATES}'
n_mp4 = len(glob.glob(f'{UAH_DIR}/**/*.mp4', recursive=True))
print('CODE_DIR:', CODE_DIR)
print('UAH_DIR :', UAH_DIR, '| số video:', n_mp4)
if n_mp4 != 40:
    print('CẢNH BÁO: UAH-DriveSet đủ phải có 40 video. Drive có thể chưa đồng bộ xong.')
for sub in ['saved_models/step_3d', 'saved_models/step_1d', 'saved_models/step_fusion', 'video_features', 'logs']:
    os.makedirs(f'{RUN_DIR}/{sub}', exist_ok=True)
print('RUN_DIR :', RUN_DIR)

In [ ]:
!pip install -q "timm>=1.0.20"
import torch, timm, cv2
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| timm', timm.__version__, '| cv2', cv2.__version__)

## 1. Dữ liệu: làm sạch UAH-DriveSet và tạo pickle

Script `prepare_uah_data.py` không sinh dữ liệu: kinematic là 51 mẫu gia tốc thật liên tiếp, video là 77 frame thật, đồng bộ bằng offset đã kiểm chứng (tương quan optical flow của video với yaw rate). Cửa sổ nào không đạt kiểm tra thì bị loại, lý do ghi trong report.

**RAM**: pipeline nạp toàn bộ video vào RAM. Mỗi mẫu float16 ~23 MB. Colab miễn phí có ~12 GB RAM, nên mặc định lấy tối đa 8 cửa sổ/chuyến (~300 mẫu). Có RAM lớn hơn thì tăng `MAX_WINDOWS_PER_TRIP`.

In [ ]:
MAX_WINDOWS_PER_TRIP = 8
VAL_DRIVERS = 'D6'
# D2-DROWSY-SECONDARY: tên video (165604) lệch 10 phút so với tên chuyến (164606), gần như chắc là gõ nhầm phút.
# Giả thuyết offset +2 s đã được kiểm chứng bằng dữ liệu: đỉnh tương quan yaw rate tại +4.4 s, corr 0.82,
# quét toàn bộ độ trễ cũng ra +4.4 s. Script vẫn kiểm chứng lại mỗi lần chạy, sai thì chuyến bị loại.
OFFSET_OVERRIDES = ['20151120164606-16km-D2-DROWSY-SECONDARY=2']
REUSE_DRIVE_DATA = True      # nếu RUN_DIR/data đã có pickle thì dùng lại, không xử lý lại
SAVE_DATA_TO_DRIVE = True    # lưu pickle vừa tạo lên Drive để lần sau dùng lại

PREP_ARGS = f'--max_windows_per_trip {MAX_WINDOWS_PER_TRIP} --val_drivers {VAL_DRIVERS} ' + ' '.join(
    f'--offset_override "{o}"' for o in OFFSET_OVERRIDES)
DATA_FILES = [f'{k}_{s}.pkl' for k in ['video', 'kine', 'label'] for s in ['train', 'val']]
have_drive_data = all(os.path.isfile(f'{DRIVE_DATA_DIR}/{f}') for f in DATA_FILES)
print('Pickle đã có trên Drive:', have_drive_data)

In [ ]:
# Chép UAH-DriveSet sang ổ local (~3.4 GB) — bỏ qua nếu dùng lại pickle trên Drive
if not (REUSE_DRIVE_DATA and have_drive_data):
    !mkdir -p {LOCAL_UAH}
    !rsync -a --info=progress2 --exclude='*~' "{UAH_DIR}/" {LOCAL_UAH}/
    print('video local:', len(glob.glob(f'{LOCAL_UAH}/**/*.mp4', recursive=True)))

In [ ]:
# Bước kiểm tra (không ghi pickle): xem chuyến nào đạt / bị loại và vì sao
import pandas as pd
if not (REUSE_DRIVE_DATA and have_drive_data):
    !cd "{CODE_DIR}" && python prepare_uah_data.py --uah_root {LOCAL_UAH} --out_dir {LOCAL_DATA} {PREP_ARGS} --check_only
    rep = pd.read_csv(f'{LOCAL_DATA}/uah_trip_report.csv')
    cols = ['trip', 'status', 'name_offset', 'offset_used', 'sync_near_corr', 'sync_near_ratio', 'sync_wide_lag',
            'windows_valid', 'windows_chosen', 'note']
    display(rep[[c for c in cols if c in rep.columns]])

In [ ]:
# Tạo pickle (hoặc lấy lại từ Drive)
import shutil
os.makedirs(LOCAL_DATA, exist_ok=True)
if REUSE_DRIVE_DATA and have_drive_data:
    for f in os.listdir(DRIVE_DATA_DIR):
        shutil.copy2(f'{DRIVE_DATA_DIR}/{f}', LOCAL_DATA)
    print('Đã chép pickle từ Drive về', LOCAL_DATA)
else:
    !cd "{CODE_DIR}" && python prepare_uah_data.py --uah_root {LOCAL_UAH} --out_dir {LOCAL_DATA} {PREP_ARGS}
    if SAVE_DATA_TO_DRIVE:
        os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
        for f in os.listdir(LOCAL_DATA):
            if not f.endswith('.tmp.npy'):
                shutil.copy2(f'{LOCAL_DATA}/{f}', DRIVE_DATA_DIR)
        print('Đã lưu pickle lên', DRIVE_DATA_DIR)
!ls -lh {LOCAL_DATA}

In [ ]:
# Kiểm tra nhanh dữ liệu trước khi train
import numpy as np
for s in ['train', 'val']:
    k = pd.read_pickle(f'{LOCAL_DATA}/kine_{s}.pkl'); l = pd.read_pickle(f'{LOCAL_DATA}/label_{s}.pkl')
    w = pd.read_csv(f'{LOCAL_DATA}/windows_{s}.csv')
    assert len(k) == len(l) == len(w), 'số mẫu kine/label/windows không khớp'
    print(s, 'mẫu:', len(l), '| nhãn (0=NORMAL,1=AGGRESSIVE,2=DROWSY):', np.bincount(l, minlength=3),
          '| tài xế:', sorted(w.driver.unique()))

## 2. Huấn luyện 4 stage

Số epoch mặc định giống project gốc. Giảm để chạy thử nhanh.

In [ ]:
EPOCHS_3D, BS_3D = 100, 9       # Stage 1 (gốc: 100 epoch, batch 9)
EPOCHS_1D, BS_1D = 200, 2000    # Stage 2 (gốc: 200 epoch, batch 2000)
EPOCHS_FU, BS_FU = 200, 2000    # Stage 4 (gốc: 200 epoch, batch 2000)

VIDEO_ARGS = (f'--path_video_train {LOCAL_DATA}/video_train.pkl --path_video_val {LOCAL_DATA}/video_val.pkl '
              f'--path_label_train {LOCAL_DATA}/label_train.pkl --path_label_val {LOCAL_DATA}/label_val.pkl')
KINE_ARGS = (f'--path_kine_train {LOCAL_DATA}/kine_train.pkl --path_kine_val {LOCAL_DATA}/kine_val.pkl '
             f'--path_label_train {LOCAL_DATA}/label_train.pkl --path_label_val {LOCAL_DATA}/label_val.pkl')

In [ ]:
# Stage 1: train DINOv3 video encoder (DINOv3 ViT-S/16 frozen, trọng số từ model.safetensors)
cmd = (f'cd "{CODE_DIR}" && python step_dinov3_3d.py {VIDEO_ARGS} --num_epochs {EPOCHS_3D} --batch_size {BS_3D} '
       f'--outpath "{RUN_DIR}/saved_models/step_3d/" --logpath "{RUN_DIR}/logs/"')
!{cmd}

In [ ]:
!grep -c "save_all_epochs" "{CODE_DIR}/step_dinov3_3d.py" "{CODE_DIR}/step_fusion_1d.py" "{CODE_DIR}/step_fusion.py"


In [ ]:
# Stage 2: train 1D Swin trên kinematic
cmd = (f'cd "{CODE_DIR}" && python step_fusion_1d.py {KINE_ARGS} --num_epochs {EPOCHS_1D} --batch_size {BS_1D} '
       f'--outpath "{RUN_DIR}/saved_models/step_1d/" --logpath "{RUN_DIR}/logs/"')
!{cmd}

In [ ]:
# Lưu thống kê chuẩn hoá cạnh model 1D, web cần đủ 2 file này
import shutil, os
dst = f'{RUN_DIR}/saved_models/step_1d'
shutil.copy2(f'{LOCAL_DATA}/kine_norm_stats.json', dst)
print(sorted(os.listdir(dst)))


In [ ]:
# Stage 3: trích video feature (N, 1024) từ model Stage 1
cmd = (f'cd "{CODE_DIR}" && python step_dinov3_3d_get_feature.py {VIDEO_ARGS} --batch_size 8 '
       f'--path_3d_model "{RUN_DIR}/saved_models/step_3d/final_3d.pth" '
       f'--outpath "{RUN_DIR}/video_features/" --logpath "{RUN_DIR}/logs/"')
!{cmd}

In [ ]:
# Stage 4: train fusion video + kinematic
cmd = (f'cd "{CODE_DIR}" && python step_fusion.py {KINE_ARGS} --num_epochs {EPOCHS_FU} --batch_size {BS_FU} '
       f'--path_1d_model "{RUN_DIR}/saved_models/step_1d/final_1d.pth" '
       f'--path_vidfeat_train "{RUN_DIR}/video_features/train_3d_features.pkl" '
       f'--path_vidfeat_val "{RUN_DIR}/video_features/val_3d_features.pkl" '
       f'--outpath "{RUN_DIR}/saved_models/step_fusion/" --logpath "{RUN_DIR}/logs/"')
!{cmd}

## 3. Kết quả

In [ ]:
for stage in ['step_3d', 'step_1d', 'step_fusion']:
    d = f'{RUN_DIR}/saved_models/{stage}'
    if os.path.isfile(f'{d}/val_acc.npy'):
        acc, loss = np.load(f'{d}/val_acc.npy'), np.load(f'{d}/val_loss.npy')
        i = int(np.argmin(loss))
        print(f'{stage:12s} val acc cuối={acc[-1]:.2f}% | cao nhất={acc.max():.2f}% | tại val loss nhỏ nhất={acc[i]:.2f}%')
    else:
        print(f'{stage:12s} chưa có kết quả')